# Байесовская сеть: погода
Узлы: Weather (0 — солнечно, 1 — облачно), Rain, Temp, Humidity (0 — нет/низкая, 1 — да/высокая).

Допущение: P(дождь | солнечно) = 0 — в условии не задана.

In [ ]:
!pip install -q pgmpy

In [ ]:
from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.factors.discrete import TabularCPD
from pgmpy.inference import VariableElimination

model = DiscreteBayesianNetwork([
    ('Weather', 'Rain'), ('Weather', 'Temp'),
    ('Rain', 'Humidity'), ('Temp', 'Humidity'),
])

names = {'Weather': ['sunny', 'cloudy'], 'Rain': ['no', 'yes'],
         'Temp': ['low', 'high'], 'Humidity': ['low', 'high']}

cpd_weather = TabularCPD('Weather', 2, [[0.4], [0.6]], state_names={'Weather': names['Weather']})

cpd_rain = TabularCPD('Rain', 2,
    [[1.0, 0.4],   # нет дождя   | sunny, cloudy
     [0.0, 0.6]],  # дождь       | sunny, cloudy
    evidence=['Weather'], evidence_card=[2],
    state_names={'Rain': names['Rain'], 'Weather': names['Weather']})

cpd_temp = TabularCPD('Temp', 2,
    [[0.2, 0.5],   # низкая      | sunny, cloudy
     [0.8, 0.5]],  # высокая     | sunny, cloudy
    evidence=['Weather'], evidence_card=[2],
    state_names={'Temp': names['Temp'], 'Weather': names['Weather']})

# столбцы: (Rain=no,Temp=low), (no,high), (yes,low), (yes,high)
cpd_hum = TabularCPD('Humidity', 2,
    [[0.7, 0.5, 0.6, 0.1],
     [0.3, 0.5, 0.4, 0.9]],
    evidence=['Rain', 'Temp'], evidence_card=[2, 2],
    state_names={'Humidity': names['Humidity'], 'Rain': names['Rain'], 'Temp': names['Temp']})

model.add_cpds(cpd_weather, cpd_rain, cpd_temp, cpd_hum)
assert model.check_model()
for c in model.get_cpds():
    print(c)

In [ ]:
infer = VariableElimination(model)
marg = {v: infer.query([v], show_progress=False) for v in model.nodes()}
for v, f in marg.items():
    print(v, {s: round(float(p), 4) for s, p in zip(f.state_names[v], f.values)})

## Визуализация (как в UnBBayes)

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import FancyArrowPatch, Rectangle

pos = {'Weather': (5, 8), 'Rain': (1.5, 5), 'Temp': (8.5, 5), 'Humidity': (5, 1.5)}
W, H = 3.2, 1.9

def draw(marginals, title='Априорные вероятности'):
    fig, ax = plt.subplots(figsize=(10, 7.5))
    ax.set_xlim(-1, 11); ax.set_ylim(-0.5, 10); ax.axis('off'); ax.set_title(title)
    for u, v in model.edges():
        (x1, y1), (x2, y2) = pos[u], pos[v]
        ax.add_patch(FancyArrowPatch((x1, y1 - H/2), (x2, y2 + H/2),
                     arrowstyle='-|>', mutation_scale=18, color='#1f3c88'))
    for n, (x, y) in pos.items():
        f = marginals[n]; states = f.state_names[n]
        ax.add_patch(Rectangle((x - W/2, y - H/2), W, H, fc='#fff36b', ec='#1f3c88', lw=1.5))
        ax.text(x, y + H/2 - 0.3, n, ha='center', va='center', fontsize=12, weight='bold')
        for i, (s, p) in enumerate(zip(states, f.values)):
            yy = y + 0.05 - i * 0.6
            ax.text(x - W/2 + 0.1, yy, s, va='center', fontsize=10)
            ax.text(x + 0.45, yy, f'{p*100:.1f}%', va='center', ha='right', fontsize=10)
            ax.add_patch(Rectangle((x + 0.55, yy - 0.2), 0.95 * p, 0.4, fc='#3b5bdb'))
            ax.add_patch(Rectangle((x + 0.55, yy - 0.2), 0.95, 0.4, fill=False, ec='gray', lw=0.6))
    plt.show()
    return fig

draw(marg);

## Пример вывода по свидетельству
P(Weather | Humidity = high) и P(Rain | Humidity = high)

In [ ]:
ev = {'Humidity': 'high'}
print(infer.query(['Weather'], evidence=ev, show_progress=False))
print(infer.query(['Rain'], evidence=ev, show_progress=False))